# MPR-Agent — graph-based multi-agent pipeline (ViNumQA)

Driver for the `agentic/` package beside this notebook: an implementation of
Nguyen et al., *"A Graph-Based Agent Approach to Numerical Reasoning Question
Answering"* ([VLSP 2025](https://aclanthology.org/2025.vlsp-1.29/)) — the system
that **won Subtask 2 of this shared task with EA 84.00%**.

Unlike every other method in this repo, it trains nothing. It is inference-only,
and restructures *how* the model is asked into a four-node agent DAG:

```
q, C ──▶ [1] SubqueryGenerator   G_sq(q, C)            SQ = {sq_1..sq_k}, k∈[3,5]
                    │ fan-out, one independent call per subquery
         [2] SubqueryAnswerer    A_sq(sq_j, C)         V  = {v_1..v_k}
                    │ fan-in
         [3] Planner             P_n-sample(V,C,q,T)   n = 15 candidate plans
                    │
         [4] EquationExtractor   canonicalise → vote → p* → Execute → a*
```

**All the logic lives in `agentic/`** — read `README.md` beside this notebook
for the design, the seven plan-DSL → ViNumQA transpilation rules, which
defaults are the paper's versus ours, and the one known fidelity gap. This
notebook only drives it and reports numbers.

**Prerequisites**: `API_KEY` and `BASE_URL` in `.env` at the project root, and
`pytest notebooks/vinumqa/graph-agent/tests -q` green.

In [ ]:
import json
import sys
import time
from dataclasses import replace
from pathlib import Path

# Resolve the project root by walking up to the nearest .git, the same way every
# other notebook here does, so this runs from the repo root or from this folder.
_here = Path.cwd()
ROOT = next((p for p in (_here, *_here.parents) if (p / ".git").exists()), None)
assert ROOT is not None, f"project root not found above {_here}"
HERE = ROOT / "notebooks" / "vinumqa" / "graph-agent"
sys.path.insert(0, str(HERE))          # so `import agentic` finds the package

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

import pandas as pd

from agentic import AgentConfig, RunConfig, Runner
from agentic.runner import candidate_diagnostics, load_dataset, revote, score_frame

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)
print(f"project root: {ROOT}")
print(f"package     : {HERE / 'agentic'}")

## Configuration

Defaults are the paper's own (§5.1): `n = 15`, `temperature = 0.6`,
`top_p = 0.95`, `top_k = 20`, Vietnamese prompts verbatim from Appendix B.

`gemma-4-31B-it` is chosen deliberately: it is this repo's strongest in-context
baseline (few-shot(3): **PA 0.5674 / EA 0.6137** on the same `test.json`), so
running the agent on the same model measures the *architecture's* contribution
rather than a model swap — the comparison the paper draws in its Table 2.

Note this is **not** comparable to the paper's own headline numbers: different
evaluation split, different model, and a different PA definition (the paper
uses normalised string identity; `scorer.py` uses sympy symbolic equivalence
with a literal restriction). The meaningful comparison is against the repo rows
below.

In [ ]:
MODEL = "gemma-4-31B-it"

agent_config = AgentConfig(
    model_subquery_gen=MODEL,
    model_subquery_ans=MODEL,
    model_planner=MODEL,
    model_fallback=MODEL,
    # --- paper section 5.1 ---
    n_samples=15,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    prompt_lang="vi",
    # --- ours; see README.md ---
    vote_mode="canonical",          # the paper's section 4.4 prose
    use_prompt_ext=False,           # keep prompts verbatim for the headline run
    max_workers_dataset=4,
    # Set rpm_limit / tpm_limit if the endpoint throttles (FPT AI Factory
    # enforces RPM=50 / TPM=100_000 on GLM-5.2, for example).
)

run_config = RunConfig(
    dataset_path="datasets/ViNumQA/origin/test.json",
    output_dir="notebooks/vinumqa/graph-agent/outputs",
    run_name=f"mpr-agent-{MODEL}",
    agent=agent_config,
)

runner = Runner(run_config)
print(runner.graph.describe())

## 1. Smoke run — 30 samples

Run this before committing to the full set. It verifies all four nodes end to
end and, more importantly, measures the real cost so the full run can be
estimated rather than guessed.

Checkpointing is per sample and resume is on, so an interruption never costs
more than the sample in flight.

In [ ]:
smoke_runner = Runner(
    RunConfig(**{**run_config.__dict__, "run_name": "smoke30", "limit": 30})
)

started = time.time()
smoke_df = smoke_runner.run()
elapsed = time.time() - started

smoke_scored, smoke_summary = smoke_runner.score(smoke_df)
usage = smoke_summary["usage"]
n = len(smoke_df)

for key, value in smoke_summary.items():
    print(f"{key:>24}: {value}")
print()
print(f"{'seconds/sample':>24}: {elapsed / n:.2f}")
print(f"{'tokens/sample':>24}: {usage['total_tokens'] / n:.0f}")
print(f"{'requests/sample':>24}: {usage['requests'] / n:.1f}")
print()
full_n = len(load_dataset(run_config.dataset_path))
print(
    f"projected for {full_n} samples: "
    f"{usage['total_tokens'] / n * full_n / 1e6:.1f}M tokens, "
    f"{usage['requests'] / n * full_n:.0f} requests, "
    f"{elapsed / n * full_n / 60:.0f} min"
)

### Read one trace end to end

The point of an agent pipeline over a single prompt is that every intermediate
step is inspectable. Read a few of these by hand before trusting any aggregate.

In [ ]:
with open(smoke_runner.checkpoint_path, encoding="utf-8") as handle:
    traces = json.load(handle)

gold_lookup = {str(s["id"]): s["qa"] for s in load_dataset(run_config.dataset_path)}


def show(record):
    gold = gold_lookup.get(record["id"], {})
    print("=" * 100)
    print(f"id       : {record['id']}")
    print(f"question : {record['question']}")
    print(f"\n[1] subqueries ({len(record['subqueries'])}):")
    for item in record["subqueries"]:
        print(f"      - {item}")
    print("\n[2] answers:")
    for item in record["subquery_answers"]:
        print(f"      * {item['answer'][:150]}")
    candidates = record.get("candidates", [])
    distinct = {c["program"] for c in candidates if c["program"]}
    print(f"\n[3] {len(candidates)} plans sampled -> {len(distinct)} distinct program(s)")
    for program in list(distinct)[:5]:
        print(f"      {program}")
    vote_info = record.get("vote") or {}
    print(
        f"\n[4] clusters={vote_info.get('n_clusters')} "
        f"consensus={vote_info.get('consensus')} fallback={record.get('fallback')}"
    )
    print(f"      p*   : {record['program']}")
    print(f"      gold : {gold.get('program')}")
    print(f"      a*   : {record['answer']}   gold: {gold.get('exe_ans')}")


for record in traces[:3]:
    show(record)

### Where candidates die

Each of the `n × samples` generated plans either becomes a program or fails at a
named stage: `parse` (not a plan), `transpile` (a plan with no ViNumQA
equivalent), `row_lookup` (`table_*` naming a row the table does not have), or
`execute`. A high count in any one of these is a prompt problem, not a model
problem, and points at which one.

In [ ]:
diagnostics = candidate_diagnostics(smoke_runner.checkpoint_path)
print(diagnostics["stage"].value_counts(normalize=True).round(4).to_string())
print("\nmost common failures:")
print(diagnostics[diagnostics["stage"] != "ok"]["error"]
      .value_counts().head(10).to_string())

### Is n-sampling actually doing anything?

The paper's headline contribution only helps if the `n` samples genuinely
differ. If every sample collapses to one distinct program, the vote has nothing
to decide and `oracle@n` will equal plain accuracy exactly — check this before
attributing any result to multi-path reasoning.

Measured here on `gemma-4-31B-it`: **26 of 30 samples produced a single distinct
plan** at the paper's `temperature = 0.6`. Verified three ways on a real planner
prompt — this package's client, a raw server-side `n=15` call, and 15 separate
requests — all returned 1 distinct plan, while a control probe on an open-ended
prompt returned 9 distinct out of 15. The planner task is simply
near-deterministic once decomposition has pinned the numbers down.
`AgentConfig(temperature_planner=...)` raises planner temperature alone.

In [ ]:
distinct_programs = [
    len({c["program"] for c in r.get("candidates", []) if c["program"]})
    for r in traces
]
distinct_plans = [len({c["raw_plan"] for c in r.get("candidates", [])}) for r in traces]
print("distinct raw plans per sample:")
print(pd.Series(distinct_plans).value_counts().sort_index().to_string())
print("\ndistinct programs per sample:")
print(pd.Series(distinct_programs).value_counts().sort_index().to_string())
print(
    f"\noracle_pa {smoke_summary['oracle_pa']:.4f} vs "
    f"PA {smoke_summary['program_accuracy']:.4f}  "
    f"(gap = what voting threw away)"
)

## 2. Full run — `test.json`, 497 samples

The same fixed evaluation set as every row in the root README, scored with the
same `notebooks/evaluate/scorer.py`, so the result is directly comparable to
them.

In [ ]:
started = time.time()
df = runner.run()
print(f"elapsed: {(time.time() - started) / 60:.1f} min")

scored, summary = runner.score(df)
results_path, summary_path = runner.save(scored, summary)

for key, value in summary.items():
    print(f"{key:>24}: {value}")
print(f"\nsaved: {results_path}\n       {summary_path}")

### Against this repo's existing baselines

Same model, same test set, same scorer — the only thing that changes is the
architecture. This mirrors the paper's Table 2.

In [ ]:
comparison = pd.DataFrame(
    [
        {"method": f"{MODEL}, 0-shot", "PA": None, "EA": None},
        {"method": f"{MODEL}, few-shot (3)", "PA": 0.5674, "EA": 0.6137},
        {
            "method": f"{MODEL}, MPR-Agent",
            "PA": round(summary["program_accuracy"], 4),
            "EA": round(summary["execution_accuracy"], 4),
        },
        {
            "method": f"{MODEL}, MPR-Agent (oracle@15)",
            "PA": round(summary["oracle_pa"], 4),
            "EA": round(summary["oracle_ea"], 4),
        },
    ]
)
# Fill the 0-shot row from that notebook's own saved summary if it is present.
zero_shot = ROOT / f"notebooks/vinumqa/0-shot/outputs/0shot_{MODEL}_summary.json"
if zero_shot.exists():
    with open(zero_shot, encoding="utf-8") as handle:
        blob = json.load(handle)
    comparison.loc[0, "PA"] = round(blob["program_accuracy"], 4)
    comparison.loc[0, "EA"] = round(blob["execution_accuracy"], 4)
comparison

### Error analysis (paper §5.5.2)

The paper splits failures into two kinds but reports no numbers for either.
`oracle@n` separates them:

* `oracle_pa − PA` — the correct program **was** generated and voting discarded
  it (their *heuristic selection error*).
* `1 − oracle_pa` — the correct program was **never generated** (their
  *systematic reasoning error*). Voting cannot help here; only a better base
  model or better prompts can.

In [ ]:
pa = summary["program_accuracy"]
oracle = summary["oracle_pa"]
print(f"correct and selected    : {pa:.4f}")
print(f"generated but out-voted : {oracle - pa:.4f}   (heuristic selection error)")
print(f"never generated         : {1 - oracle:.4f}   (systematic reasoning error)")
print(f"\nfallback rate           : {summary['fallback_rate']:.4f}")
print(f"empty predictions       : {summary['empty_rate']:.4f}")

# High consensus on a wrong answer is the signature of a systematic error.
wrong = scored[(scored["pa_score"] == 0) & scored["consensus"].notna()]
if len(wrong):
    print(f"\nmean consensus on PA-wrong samples: {wrong['consensus'].mean():.4f}")
    print(f"of which unanimous (consensus 1.0): {(wrong['consensus'] == 1.0).mean():.4f}")

## 3. Re-vote offline — free, no API calls

Every candidate is kept on disk with its program and executed value
(`keep_all_candidates`), so changing *how the winner is chosen* costs nothing:
the n plans are already there. Use this to compare vote modes on an existing
run instead of paying for the whole pipeline again.

See the README's *"Known gap"* section for why a result-level mode is still
outstanding — and note that on this model, with ~1 distinct program per sample,
no vote mode can change the outcome.

In [ ]:
samples = load_dataset(run_config.dataset_path)
rows = []
for mode in ("canonical", "symbolic"):
    _, mode_summary = score_frame(revote(runner.checkpoint_path, samples, mode=mode))
    rows.append(
        {
            "vote_mode": mode,
            "PA": round(mode_summary["program_accuracy"], 4),
            "EA": round(mode_summary["execution_accuracy"], 4),
        }
    )
pd.DataFrame(rows)

## 4. Ablation — reproducing the paper's Table 4

Three configurations, one flag each. The direct-prompt baseline row already
exists in this repo (the 0-shot / few-shot notebooks), so it is not re-run.

| Configuration | Flag |
|---|---|
| Full MPR-Agent | defaults |
| Decomposition only | `n_samples=1` |
| Multi-path only | `use_decomposition=False` |

**Cost warning**: each is a full pass over 497 samples.

In [ ]:
ABLATIONS = {
    "decomposition-only": replace(agent_config, n_samples=1),
    "multi-path-only": replace(agent_config, use_decomposition=False),
}

rows = [
    {
        "configuration": "full MPR-Agent",
        "EA": round(summary["execution_accuracy"], 4),
        "PA": round(summary["program_accuracy"], 4),
    }
]
for name, config in ABLATIONS.items():
    ablation_runner = Runner(
        RunConfig(
            dataset_path=run_config.dataset_path,
            output_dir=run_config.output_dir,
            run_name=f"mpr-agent-{MODEL}-{name}",
            agent=config,
        )
    )
    ablation_scored, ablation_summary = ablation_runner.score(ablation_runner.run())
    ablation_runner.save(ablation_scored, ablation_summary)
    rows.append(
        {
            "configuration": name,
            "EA": round(ablation_summary["execution_accuracy"], 4),
            "PA": round(ablation_summary["program_accuracy"], 4),
        }
    )

pd.DataFrame(rows)

## 5. Optional — the prompt-fidelity A/B

Appendix B.9/B.10 tells the planner to strip a percent sign and use `48.8`. But
ViNumQA gold writes a percentage *used as a rate* as a decimal —
`divide(333, 0.159)` — and **34 of the 497 gold test programs** contain such a
literal. Following the prompt literally produces
`divide(15.9, 100), divide(333, #0)`: **EA passes, PA fails**, because
`scorer.py`'s symbolic PA admits only literals present in gold.

That is the paper's own Example 5.1, and the likely reason its PA (74.07) trails
its EA (84.00). `use_prompt_ext=True` adds one bullet telling the planner to
write the decimal directly. Run it to measure the cost of prompt fidelity
instead of assuming it.

In [ ]:
ext_runner = Runner(
    RunConfig(
        dataset_path=run_config.dataset_path,
        output_dir=run_config.output_dir,
        run_name=f"mpr-agent-{MODEL}-promptext",
        agent=replace(agent_config, use_prompt_ext=True),
    )
)
ext_scored, ext_summary = ext_runner.score(ext_runner.run())
ext_runner.save(ext_scored, ext_summary)

pd.DataFrame(
    [
        {
            "prompts": "verbatim (paper)",
            "EA": round(summary["execution_accuracy"], 4),
            "PA": round(summary["program_accuracy"], 4),
        },
        {
            "prompts": "+ percent-as-decimal patch",
            "EA": round(ext_summary["execution_accuracy"], 4),
            "PA": round(ext_summary["program_accuracy"], 4),
        },
    ]
)